In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv(r"C:\Users\DeLL\Desktop\fake-bills-ml-project\data\fake_bills.csv", sep = ";")

In [ ]:
# from google.colab import files
# uploaded = files.upload()

Saving fake_bills.csv to fake_bills.csv


In [3]:
df = pd.read_csv("fake_bills.csv",sep = ";")

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   is_genuine    1500 non-null   bool   
 1   diagonal      1500 non-null   float64
 2   height_left   1500 non-null   float64
 3   height_right  1500 non-null   float64
 4   margin_low    1463 non-null   float64
 5   margin_up     1500 non-null   float64
 6   length        1500 non-null   float64
dtypes: bool(1), float64(6)
memory usage: 71.9 KB


In [5]:
(df.isnull().sum()/len(df))*100

,0
is_genuine,0.000000
diagonal,0.000000
height_left,0.000000
height_right,0.000000
margin_low,2.466667
margin_up,0.000000
length,0.000000


In [3]:
df.dropna(inplace = True)

In [7]:
df.head()

,is_genuine,diagonal,height_left,height_right,margin_low,margin_up,length
0,True,171.81,104.86,104.95,4.52,2.89,112.83
1,True,171.46,103.36,103.66,3.77,2.99,113.09
2,True,172.69,104.48,103.50,4.40,2.94,113.16
3,True,171.36,103.91,103.94,3.62,3.01,113.51
4,True,171.73,104.28,103.46,4.04,3.48,112.54


is_genuine - Target


In [4]:
numerical_col = ["diagonal",	"height_left","height_right",	"margin_low",	"margin_up",	"length"]

In [9]:
for col in numerical_col:
  print(f"skewness of {col}:- \n {df[col].skew()}")

skewness of diagonal:- 
 -0.04156253039005744
skewness of height_left:- 
 -0.09587063501262938
skewness of height_right:- 
 0.013368648746703135
skewness of margin_low:- 
 0.8630655172248257
skewness of margin_up:- 
 0.1489687662289017
skewness of length:- 
 -0.8133874519466804


length - left skewed,
margin_low - right skewed

In [5]:
skewed_col = ["length","margin_low"]

In [6]:
normal_num_col = ["diagonal",	"height_left","height_right",	"margin_up"]

In [7]:
X, y = df.drop(columns = ["is_genuine"]), df["is_genuine"]

In [8]:
from sklearn.model_selection import train_test_split

In [9]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 13)

In [10]:
from sklearn.pipeline import Pipeline

In [11]:
from sklearn.preprocessing import PowerTransformer

In [12]:
from sklearn.preprocessing import StandardScaler

In [13]:
skewed_pipe = Pipeline([
    ("power", PowerTransformer()),
    ("scaler", StandardScaler())
])

In [14]:
num_pipe = Pipeline([
    ("scaler", StandardScaler())
])

In [15]:
from sklearn.preprocessing import RobustScaler
num_pipe_robust = Pipeline([
    ("scaler", RobustScaler())
])

In [17]:
from sklearn.compose import ColumnTransformer

In [18]:
preprocessor = ColumnTransformer([
    ("skewed", skewed_pipe, skewed_col),
    ("normal", num_pipe, normal_num_col) ])

In [19]:
preprocessor_no_transform_base = ColumnTransformer([
    ("numerical", num_pipe_robust, numerical_col)
])

In [20]:
y.value_counts()

is_genuine
True     971
False    492
Name: count, dtype: int64

# Base-Line Model

In [21]:
from sklearn.linear_model import LogisticRegression

In [22]:
baseline_model = LogisticRegression(max_iter = 1000,
                                    solver="lbfgs",
                                    class_weight="balanced",
                                    random_state = 13)

In [23]:
base_pipe = Pipeline(
    [
     ("preprocessor", preprocessor),
     ("model", baseline_model)
    ]
)

In [24]:
base_pipe_no_transform = Pipeline(
    [
     ("preprocessor", preprocessor_no_transform_base),
     ("model", baseline_model)
    ])

In [25]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1463 entries, 0 to 1499
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   is_genuine    1463 non-null   bool   
 1   diagonal      1463 non-null   float64
 2   height_left   1463 non-null   float64
 3   height_right  1463 non-null   float64
 4   margin_low    1463 non-null   float64
 5   margin_up     1463 non-null   float64
 6   length        1463 non-null   float64
dtypes: bool(1), float64(6)
memory usage: 81.4 KB


In [26]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1170 entries, 16 to 346
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   diagonal      1170 non-null   float64
 1   height_left   1170 non-null   float64
 2   height_right  1170 non-null   float64
 3   margin_low    1170 non-null   float64
 4   margin_up     1170 non-null   float64
 5   length        1170 non-null   float64
dtypes: float64(6)
memory usage: 64.0 KB


In [27]:
base_pipe.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('skewed', ...), ('normal', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [28]:
base_pipe_no_transform.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('numerical', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [29]:
y_pred_base = base_pipe.predict(X_test)

In [30]:
y_pred_base_no_transform = base_pipe_no_transform.predict(X_test)

In [31]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

In [32]:
print(f"Accuracy: {accuracy_score(y_test, y_pred_base)}")
print(f"Precision : {precision_score(y_test, y_pred_base, pos_label=False)}")
print(f"Recall : {recall_score(y_test, y_pred_base, pos_label=False)}")
print(f"F1 Score : {f1_score(y_test, y_pred_base, pos_label=False)}")
print(f"Classification Report:\n{classification_report(y_test, y_pred_base)}")
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred_base)
print(f"confusion matrix :\n{cm}")

Accuracy: 0.5187713310580204
Precision : 0.1839080459770115
Recall : 0.18604651162790697
F1 Score : 0.18497109826589594
Classification Report:
              precision    recall  f1-score   support

       False       0.18      0.19      0.18        86
        True       0.66      0.66      0.66       207

    accuracy                           0.52       293
   macro avg       0.42      0.42      0.42       293
weighted avg       0.52      0.52      0.52       293

confusion matrix :
[[ 16  70]
 [ 71 136]]


In [33]:
print(f"Accuracy: {accuracy_score(y_test, y_pred_base_no_transform)}")
print(f"Precision : {precision_score(y_test, y_pred_base_no_transform, pos_label=False)}")
print(f"Recall : {recall_score(y_test, y_pred_base_no_transform, pos_label=False)}")
print(f"F1 Score : {f1_score(y_test, y_pred_base_no_transform, pos_label=False)}")
print(f"Classification Report:\n{classification_report(y_test, y_pred_base_no_transform)}")
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred_base_no_transform)
print(f"confusion matrix :\n{cm}")

Accuracy: 0.9931740614334471
Precision : 0.9883720930232558
Recall : 0.9883720930232558
F1 Score : 0.9883720930232558
Classification Report:
              precision    recall  f1-score   support

       False       0.99      0.99      0.99        86
        True       1.00      1.00      1.00       207

    accuracy                           0.99       293
   macro avg       0.99      0.99      0.99       293
weighted avg       0.99      0.99      0.99       293

confusion matrix :
[[ 85   1]
 [  1 206]]


In [34]:
"""
 [[TN  FP]
 [FN  TP]]
 """

'\n [[TN  FP]\n [FN  TP]]\n '

# Logistic Regression with Parameter Tuning using SMOTE

In [35]:
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

In [36]:
smote = SMOTE(random_state = 13)

In [37]:
logistic = LogisticRegression()

In [38]:
params = {
    "logistic__C": [0.01, 0.1, 1, 10],
    "logistic__penalty": ["l2"],
    "logistic__solver": ["lbfgs"]
}

In [39]:
preprocessor_logistic = ColumnTransformer([
    ("skewed", skewed_pipe, skewed_col),
    ("normal", num_pipe, normal_num_col)])

In [40]:
preprocessor_logistic_no_transform = ColumnTransformer([
    ("numerical", num_pipe_robust, numerical_col)
])

In [41]:
pipeline_logistic = ImbPipeline([
    ("smote", SMOTE(random_state=42)),
    ("logistic", LogisticRegression(max_iter=1000))
])

In [42]:
pipe_logistic_with_scaler = ImbPipeline([
    ("smote", SMOTE(random_state=42)),
    ("preprocessing",preprocessor_logistic),
    ("logistic", LogisticRegression(max_iter=1000))
])

In [43]:
pipe_logistic_no_transform = ImbPipeline([
    ("smote", SMOTE(random_state=42)),
    ("preprocessing",preprocessor_logistic_no_transform),
    ("logistic", logistic)
])

In [44]:
from sklearn.model_selection import GridSearchCV

In [45]:
grid_logistic = GridSearchCV(pipeline_logistic, params, cv=5, scoring="f1", n_jobs=-1, verbose=1)

In [46]:
grid_logistic_scaler = GridSearchCV(pipe_logistic_with_scaler, params, cv=5, scoring="f1", n_jobs=-1, verbose=1)

In [47]:
grid_logistic_no_transform = GridSearchCV(pipe_logistic_no_transform, params, cv = 5, scoring = "f1", n_jobs = -1, verbose = 2)

In [48]:
grid_logistic.fit(X_train, y_train)

Fitting 5 folds for each of 4 candidates, totalling 20 fits


,estimator,Pipeline(step..._iter=1000))])
,param_grid,"{'logistic__C': [0.01, 0.1, ...], 'logistic__penalty': ['l2'], 'logistic__solver': ['lbfgs']}"
,scoring,'f1'
,n_jobs,-1
,refit,True
,cv,5
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,sampling_strategy,'auto'


In [49]:
grid_logistic_scaler.fit(X_train, y_train)

Fitting 5 folds for each of 4 candidates, totalling 20 fits


,estimator,Pipeline(step..._iter=1000))])
,param_grid,"{'logistic__C': [0.01, 0.1, ...], 'logistic__penalty': ['l2'], 'logistic__solver': ['lbfgs']}"
,scoring,'f1'
,n_jobs,-1
,refit,True
,cv,5
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,sampling_strategy,'auto'


In [50]:
grid_logistic_no_transform.fit(X_train, y_train)

Fitting 5 folds for each of 4 candidates, totalling 20 fits


,estimator,Pipeline(step...egression())])
,param_grid,"{'logistic__C': [0.01, 0.1, ...], 'logistic__penalty': ['l2'], 'logistic__solver': ['lbfgs']}"
,scoring,'f1'
,n_jobs,-1
,refit,True
,cv,5
,verbose,2
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,sampling_strategy,'auto'


In [51]:
best_logistic = grid_logistic.best_estimator_

In [52]:
best_logistic_scaler = grid_logistic_scaler.best_estimator_

In [53]:
best_logistic_no_transform = grid_logistic_no_transform.best_estimator_

In [54]:
y_pred_logistic = grid_logistic.predict(X_test)

In [55]:
y_pred_logistic_scaler = grid_logistic_scaler.predict(X_test)

In [56]:
y_pred_logistic_no_transform = grid_logistic_no_transform.predict(X_test)

In [57]:
print(f"Accuracy: {accuracy_score(y_test, y_pred_logistic)}")
print(f"Precision : {precision_score(y_test, y_pred_logistic, pos_label=False)}")
print(f"Recall : {recall_score(y_test, y_pred_logistic, pos_label=False)}")
print(f"F1 Score : {f1_score(y_test, y_pred_logistic, pos_label=False)}")
print(f"Classification Report:\n{classification_report(y_test, y_pred_logistic)}")
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred_logistic)
print(f"confusion matrix :\n{cm}")

Accuracy: 0.9931740614334471
Precision : 0.9883720930232558
Recall : 0.9883720930232558
F1 Score : 0.9883720930232558
Classification Report:
              precision    recall  f1-score   support

       False       0.99      0.99      0.99        86
        True       1.00      1.00      1.00       207

    accuracy                           0.99       293
   macro avg       0.99      0.99      0.99       293
weighted avg       0.99      0.99      0.99       293

confusion matrix :
[[ 85   1]
 [  1 206]]


In [58]:
print(f"Accuracy: {accuracy_score(y_test, y_pred_logistic_scaler)}")
print(f"Precision : {precision_score(y_test, y_pred_logistic_scaler, pos_label=False)}")
print(f"Recall : {recall_score(y_test, y_pred_logistic_scaler, pos_label=False)}")
print(f"F1 Score : {f1_score(y_test, y_pred_logistic_scaler, pos_label=False)}")
print(f"Classification Report:\n{classification_report(y_test, y_pred_logistic_scaler)}")
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred_logistic_scaler)
print(f"confusion matrix :\n{cm}")

Accuracy: 0.9931740614334471
Precision : 0.9883720930232558
Recall : 0.9883720930232558
F1 Score : 0.9883720930232558
Classification Report:
              precision    recall  f1-score   support

       False       0.99      0.99      0.99        86
        True       1.00      1.00      1.00       207

    accuracy                           0.99       293
   macro avg       0.99      0.99      0.99       293
weighted avg       0.99      0.99      0.99       293

confusion matrix :
[[ 85   1]
 [  1 206]]


In [59]:
print(f"Accuracy: {accuracy_score(y_test, y_pred_logistic_no_transform)}")
print(f"Precision : {precision_score(y_test, y_pred_logistic_no_transform, pos_label=False)}")
print(f"Recall : {recall_score(y_test, y_pred_logistic_no_transform, pos_label=False)}")
print(f"F1 Score : {f1_score(y_test, y_pred_logistic_no_transform, pos_label=False)}")
print(f"Classification Report:\n{classification_report(y_test, y_pred_logistic_no_transform)}")
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred_logistic_no_transform)
print(f"confusion matrix :\n{cm}")

Accuracy: 0.9931740614334471
Precision : 0.9883720930232558
Recall : 0.9883720930232558
F1 Score : 0.9883720930232558
Classification Report:
              precision    recall  f1-score   support

       False       0.99      0.99      0.99        86
        True       1.00      1.00      1.00       207

    accuracy                           0.99       293
   macro avg       0.99      0.99      0.99       293
weighted avg       0.99      0.99      0.99       293

confusion matrix :
[[ 85   1]
 [  1 206]]


In [60]:
"""
 [[TN  FP]
 [FN  TP]]
 """

'\n [[TN  FP]\n [FN  TP]]\n '

# RandomForestClassifier

In [61]:
from sklearn.ensemble import RandomForestClassifier

In [62]:
rfc = RandomForestClassifier(
    random_state=42,
    bootstrap=True
)

In [63]:
preprocessor_tree = ColumnTransformer([
("skew", skewed_pipe, skewed_col),
])

In [64]:
pipeline_rfc = ImbPipeline([
("somte", smote),
("preprocessing", preprocessor_tree),
("rfc", rfc)
])

In [65]:
pipeline_rfc_no_transform = ImbPipeline([
    ("smote", smote),
    ("rfc", rfc)])

In [66]:
param_grid = {
    "rfc__n_estimators": [100, 200, 300],
    "rfc__max_depth": [None, 5, 10, 20],
    "rfc__min_samples_split": [2, 5, 10],
    "rfc__min_samples_leaf": [1, 2, 4],
    "rfc__max_features": ["sqrt", "log2"],
}

In [67]:
grid_rfc = GridSearchCV(
    estimator=pipeline_rfc,
    param_grid=param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1,
    verbose=2
)

In [68]:
grid_rfc_no_transform = GridSearchCV(pipeline_rfc_no_transform, param_grid, cv = 5,scoring = "f1", n_jobs= -1, verbose = 2)

In [69]:
grid_rfc.fit(X_train, y_train)

Fitting 5 folds for each of 216 candidates, totalling 1080 fits


,estimator,Pipeline(step...m_state=42))])
,param_grid,"{'rfc__max_depth': [None, 5, ...], 'rfc__max_features': ['sqrt', 'log2'], 'rfc__min_samples_leaf': [1, 2, ...], 'rfc__min_samples_split': [2, 5, ...], ...}"
,scoring,'f1'
,n_jobs,-1
,refit,True
,cv,5
,verbose,2
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,sampling_strategy,'auto'


In [70]:
grid_rfc_no_transform.fit(X_train, y_train)

Fitting 5 folds for each of 216 candidates, totalling 1080 fits


,estimator,Pipeline(step...m_state=42))])
,param_grid,"{'rfc__max_depth': [None, 5, ...], 'rfc__max_features': ['sqrt', 'log2'], 'rfc__min_samples_leaf': [1, 2, ...], 'rfc__min_samples_split': [2, 5, ...], ...}"
,scoring,'f1'
,n_jobs,-1
,refit,True
,cv,5
,verbose,2
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,sampling_strategy,'auto'


In [71]:
print("Best Params:", grid_rfc.best_params_)
print("Best Score:", grid_rfc.best_score_)

Best Params: {'rfc__max_depth': 5, 'rfc__max_features': 'sqrt', 'rfc__min_samples_leaf': 1, 'rfc__min_samples_split': 2, 'rfc__n_estimators': 200}
Best Score: 0.988247444312675


In [72]:
print("Best Params:", grid_rfc_no_transform.best_params_)
print("Best Score:", grid_rfc_no_transform.best_score_)

Best Params: {'rfc__max_depth': 5, 'rfc__max_features': 'sqrt', 'rfc__min_samples_leaf': 1, 'rfc__min_samples_split': 10, 'rfc__n_estimators': 100}
Best Score: 0.9934722760856793


In [73]:
best_rfc = grid_rfc.best_estimator_

In [74]:
best_rfc_no_transform = grid_rfc_no_transform.best_estimator_

In [75]:
y_pred = best_rfc.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred)}")
print(f"Precision : {precision_score(y_test, y_pred, pos_label=False)}")
print(f"Recall : {recall_score(y_test, y_pred, pos_label=False)}")
print(f"F1 Score : {f1_score(y_test, y_pred, pos_label=False)}")
print(f"Classification Report:\n{classification_report(y_test, y_pred)}")
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred)
print(f"confusion matrix :\n{cm}")

Accuracy: 0.9897610921501706
Precision : 0.9770114942528736
Recall : 0.9883720930232558
F1 Score : 0.9826589595375722
Classification Report:
              precision    recall  f1-score   support

       False       0.98      0.99      0.98        86
        True       1.00      0.99      0.99       207

    accuracy                           0.99       293
   macro avg       0.99      0.99      0.99       293
weighted avg       0.99      0.99      0.99       293

confusion matrix :
[[ 85   1]
 [  2 205]]


In [76]:
y_pred_rfc_no_transform = best_rfc_no_transform.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred_rfc_no_transform)}")
print(f"Precision : {precision_score(y_test, y_pred_rfc_no_transform, pos_label=False)}")
print(f"Recall : {recall_score(y_test, y_pred_rfc_no_transform, pos_label=False)}")
print(f"F1 Score : {f1_score(y_test, y_pred_rfc_no_transform, pos_label=False)}")
print(f"Classification Report:\n{classification_report(y_test, y_pred_rfc_no_transform)}")
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred_rfc_no_transform)
print(f"confusion matrix :\n{cm}")

Accuracy: 0.9897610921501706
Precision : 0.9770114942528736
Recall : 0.9883720930232558
F1 Score : 0.9826589595375722
Classification Report:
              precision    recall  f1-score   support

       False       0.98      0.99      0.98        86
        True       1.00      0.99      0.99       207

    accuracy                           0.99       293
   macro avg       0.99      0.99      0.99       293
weighted avg       0.99      0.99      0.99       293

confusion matrix :
[[ 85   1]
 [  2 205]]


# Gradient Boosting Classifier

In [77]:
from sklearn.ensemble import GradientBoostingClassifier

In [78]:
param_grid = {
    "gbc__n_estimators": [100, 200],
    "gbc__learning_rate": [0.05, 0.1],
    "gbc__max_depth": [3, 5],
    "gbc__min_samples_split": [2, 5],
    "gbc__min_samples_leaf": [1, 2],
    "gbc__subsample": [0.8, 1.0]   # stochastic boosting
}

In [79]:
gbc = GradientBoostingClassifier(random_state=42)

In [80]:
pipe_gbc = ImbPipeline([
    ("smote", smote),
    ("preprocessing",preprocessor_tree),
    ("gbc",gbc)])

In [81]:
pipe_gbc_no_transform = ImbPipeline([
    ("smote", smote),
    ("gbc",gbc)
])

In [82]:
grid_gbc = GridSearchCV(
    estimator=pipe_gbc,
    param_grid=param_grid,
    cv=5,
    scoring="f1",   # best for imbalance
    n_jobs=-1,
    verbose=2
)

In [83]:
grid_gbc_no_transform = GridSearchCV(
    pipe_gbc_no_transform,
    param_grid=param_grid,
    cv=5,
    scoring="f1",
    n_jobs = -1,
    verbose = 2
)

In [84]:
grid_gbc.fit(X_train, y_train)

Fitting 5 folds for each of 64 candidates, totalling 320 fits


,estimator,Pipeline(step...m_state=42))])
,param_grid,"{'gbc__learning_rate': [0.05, 0.1], 'gbc__max_depth': [3, 5], 'gbc__min_samples_leaf': [1, 2], 'gbc__min_samples_split': [2, 5], ...}"
,scoring,'f1'
,n_jobs,-1
,refit,True
,cv,5
,verbose,2
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,sampling_strategy,'auto'


In [85]:
grid_gbc_no_transform.fit(X_train, y_train)

Fitting 5 folds for each of 64 candidates, totalling 320 fits


,estimator,Pipeline(step...m_state=42))])
,param_grid,"{'gbc__learning_rate': [0.05, 0.1], 'gbc__max_depth': [3, 5], 'gbc__min_samples_leaf': [1, 2], 'gbc__min_samples_split': [2, 5], ...}"
,scoring,'f1'
,n_jobs,-1
,refit,True
,cv,5
,verbose,2
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,sampling_strategy,'auto'


In [86]:
print("Best Parameters:", grid_gbc.best_params_)
print("Best CV Score (F1):", grid_gbc.best_score_)

Best Parameters: {'gbc__learning_rate': 0.05, 'gbc__max_depth': 3, 'gbc__min_samples_leaf': 2, 'gbc__min_samples_split': 5, 'gbc__n_estimators': 200, 'gbc__subsample': 0.8}
Best CV Score (F1): 0.987583035766557


In [87]:
print("Best Parameters:", grid_gbc_no_transform.best_params_)
print("Best CV Score (F1):", grid_gbc_no_transform.best_score_)

Best Parameters: {'gbc__learning_rate': 0.1, 'gbc__max_depth': 5, 'gbc__min_samples_leaf': 1, 'gbc__min_samples_split': 5, 'gbc__n_estimators': 200, 'gbc__subsample': 0.8}
Best CV Score (F1): 0.9927950493459251


In [88]:
best_gbc = grid_gbc.best_estimator_

In [89]:
best_gbc_no_transform = grid_gbc_no_transform.best_estimator_

In [90]:
y_pred = best_gbc.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred)}")
print(f"Precision : {precision_score(y_test, y_pred, pos_label=False)}")
print(f"Recall : {recall_score(y_test, y_pred, pos_label=False)}")
print(f"F1 Score : {f1_score(y_test, y_pred, pos_label=False)}")
print(f"Classification Report:\n{classification_report(y_test, y_pred)}")
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred)
print(f"confusion matrix :\n{cm}")

Accuracy: 0.9931740614334471
Precision : 0.9772727272727273
Recall : 1.0
F1 Score : 0.9885057471264368
Classification Report:
              precision    recall  f1-score   support

       False       0.98      1.00      0.99        86
        True       1.00      0.99      1.00       207

    accuracy                           0.99       293
   macro avg       0.99      1.00      0.99       293
weighted avg       0.99      0.99      0.99       293

confusion matrix :
[[ 86   0]
 [  2 205]]


In [91]:
y_pred_gbc_no_transform = best_gbc_no_transform.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred_gbc_no_transform)}")
print(f"Precision : {precision_score(y_test, y_pred_gbc_no_transform, pos_label=False)}")
print(f"Recall : {recall_score(y_test, y_pred_gbc_no_transform, pos_label=False)}")
print(f"F1 Score : {f1_score(y_test, y_pred_gbc_no_transform, pos_label=False)}")
print(f"Classification Report:\n{classification_report(y_test, y_pred_gbc_no_transform)}")
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred_gbc_no_transform)
print(f"confusion matrix :\n{cm}")

Accuracy: 0.9897610921501706
Precision : 0.9770114942528736
Recall : 0.9883720930232558
F1 Score : 0.9826589595375722
Classification Report:
              precision    recall  f1-score   support

       False       0.98      0.99      0.98        86
        True       1.00      0.99      0.99       207

    accuracy                           0.99       293
   macro avg       0.99      0.99      0.99       293
weighted avg       0.99      0.99      0.99       293

confusion matrix :
[[ 85   1]
 [  2 205]]


# XgBoosting Classifier

In [92]:
from xgboost import XGBClassifier

In [93]:
param_grid = {
    "xgb__n_estimators": [100, 200],
    "xgb__learning_rate": [0.05, 0.1],
    "xgb__max_depth": [3, 5],
    "xgb__gamma": [0, 0.1, 0.3],
    "xgb__reg_lambda": [1, 5, 10],
    "xgb__subsample": [0.8, 1.0],
    "xgb__colsample_bytree": [0.8, 1.0]
}

In [94]:
xgb = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

In [95]:
y = y.astype(int)

In [96]:
pipeline_xgb = ImbPipeline([
    ("smote", smote),
    ("preprocessing", preprocessor_tree),
    ("xgb", xgb)
])

In [97]:
pipeline_xgb_no_transform = ImbPipeline([
    ("smote", smote),
    ("xgb", xgb)
])

In [98]:
grid_xgb = GridSearchCV(
    estimator=pipeline_xgb,
    param_grid=param_grid,
    cv=5,
    scoring="f1",   # important for imbalance
    n_jobs=-1,
    verbose=2
)

In [99]:
grid_xgb_no_transform = GridSearchCV(
    pipeline_xgb_no_transform,
    param_grid=param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1,
    verbose=2
)

In [100]:
grid_xgb.fit(X_train, y_train)

Fitting 5 folds for each of 288 candidates, totalling 1440 fits


,estimator,"Pipeline(step...=None, ...))])"
,param_grid,"{'xgb__colsample_bytree': [0.8, 1.0], 'xgb__gamma': [0, 0.1, ...], 'xgb__learning_rate': [0.05, 0.1], 'xgb__max_depth': [3, 5], ...}"
,scoring,'f1'
,n_jobs,-1
,refit,True
,cv,5
,verbose,2
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,sampling_strategy,'auto'


In [101]:
grid_xgb_no_transform.fit(X_train, y_train)

Fitting 5 folds for each of 288 candidates, totalling 1440 fits


,estimator,"Pipeline(step...=None, ...))])"
,param_grid,"{'xgb__colsample_bytree': [0.8, 1.0], 'xgb__gamma': [0, 0.1, ...], 'xgb__learning_rate': [0.05, 0.1], 'xgb__max_depth': [3, 5], ...}"
,scoring,'f1'
,n_jobs,-1
,refit,True
,cv,5
,verbose,2
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,sampling_strategy,'auto'


In [102]:
print("Best Parameters:", grid_xgb.best_params_)
print("Best CV Score (F1):", grid_xgb.best_score_)
best_xgb = grid_xgb.best_estimator_

Best Parameters: {'xgb__colsample_bytree': 0.8, 'xgb__gamma': 0.1, 'xgb__learning_rate': 0.05, 'xgb__max_depth': 5, 'xgb__n_estimators': 200, 'xgb__reg_lambda': 1, 'xgb__subsample': 1.0}
Best CV Score (F1): 0.9863014511449721


In [103]:
print("Best Parameters:", grid_xgb_no_transform.best_params_)
print("Best CV Score (F1):", grid_xgb_no_transform.best_score_)
best_xgb_no_transform = grid_xgb_no_transform.best_estimator_

Best Parameters: {'xgb__colsample_bytree': 1.0, 'xgb__gamma': 0.1, 'xgb__learning_rate': 0.1, 'xgb__max_depth': 5, 'xgb__n_estimators': 200, 'xgb__reg_lambda': 10, 'xgb__subsample': 0.8}
Best CV Score (F1): 0.9934551848811495


In [104]:
y_pred_xgb = best_xgb.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred_xgb)}")
print(f"Precision : {precision_score(y_test, y_pred_xgb, pos_label=False)}")
print(f"Recall : {recall_score(y_test, y_pred_xgb, pos_label=False)}")
print(f"F1 Score : {f1_score(y_test, y_pred_xgb, pos_label=False)}")
print(f"Classification Report:\n{classification_report(y_test, y_pred_xgb)}")
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred_xgb)
print(f"confusion matrix :\n{cm}")

Accuracy: 0.9897610921501706
Precision : 0.9662921348314607
Recall : 1.0
F1 Score : 0.9828571428571429
Classification Report:
              precision    recall  f1-score   support

       False       0.97      1.00      0.98        86
        True       1.00      0.99      0.99       207

    accuracy                           0.99       293
   macro avg       0.98      0.99      0.99       293
weighted avg       0.99      0.99      0.99       293

confusion matrix :
[[ 86   0]
 [  3 204]]


In [105]:
y_pred_no_transform = best_xgb_no_transform.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred_no_transform)}")
print(f"Precision : {precision_score(y_test, y_pred_no_transform, pos_label=False)}")
print(f"Recall : {recall_score(y_test, y_pred_no_transform, pos_label=False)}")
print(f"F1 Score : {f1_score(y_test, y_pred_no_transform, pos_label=False)}")
print(f"Classification Report:\n{classification_report(y_test, y_pred_no_transform)}")
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred_no_transform)
print(f"confusion matrix :\n{cm}")

Accuracy: 0.9931740614334471
Precision : 0.9883720930232558
Recall : 0.9883720930232558
F1 Score : 0.9883720930232558
Classification Report:
              precision    recall  f1-score   support

       False       0.99      0.99      0.99        86
        True       1.00      1.00      1.00       207

    accuracy                           0.99       293
   macro avg       0.99      0.99      0.99       293
weighted avg       0.99      0.99      0.99       293

confusion matrix :
[[ 85   1]
 [  1 206]]


In [106]:
from sklearn.model_selection import cross_val_score

In [107]:
models = [base_pipe_no_transform,best_logistic_no_transform, best_xgb_no_transform, best_rfc_no_transform, best_gbc_no_transform]

In [108]:
for model in models:
  cv = cross_val_score(model, X, y, cv=5, scoring='f1')
  print(f"cv_score for {model} is {cv.mean()}")

cv_score for Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numerical',
                                                  Pipeline(steps=[('scaler',
                                                                   RobustScaler())]),
                                                  ['diagonal', 'height_left',
                                                   'height_right', 'margin_low',
                                                   'margin_up', 'length'])])),
                ('model',
                 LogisticRegression(class_weight='balanced', max_iter=1000,
                                    random_state=13))]) is 0.9922821313975799
cv_score for Pipeline(steps=[('smote', SMOTE(random_state=42)),
                ('preprocessing',
                 ColumnTransformer(transformers=[('numerical',
                                                  Pipeline(steps=[('scaler',
                                                                   Robus

In [109]:
df.shape

(1463, 7)

# Best Model Logistic Regression without Log Transform and Robust Scaling

In [110]:
import joblib
joblib.dump(best_logistic_no_transform, "fake_bill_detector.pkl")

['fake_bill_detector.pkl']

In [192]:
from google.colab import files
files.download("fake_bill_detector.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [193]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [194]:
joblib.dump(best_logistic_no_transform, "/content/drive/MyDrive/model.pkl")

['/content/drive/MyDrive/model.pkl']

In [195]:
files.download("/content/drive/MyDrive/model.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>